# FuzzyDiff on SD3-Medium

Settings: GPU T4 x2, Internet on. Add a Kaggle Secret named `HF_TOKEN`, from a Hugging Face account that has accepted the SD3-Medium licence.

All logic lives in `code/`. Results from this notebook use provisional calibration: they show the pipeline runs and are never paper evidence.


In [ ]:
BRANCH = 'fuzzysd3'
OUT = '/kaggle/working/outputs'
RESUME = ''  # path to an earlier session's outputs, attached as a Kaggle Dataset
VAE_DEVICE = 'cuda:1'  # the transformer stays on cuda:0

!git clone -q --branch {BRANCH} https://github.com/sunzidulislam/fuzzydiff.git /kaggle/working/fuzzydiff
%cd /kaggle/working/fuzzydiff
!pip install -q -r requirements-kaggle.txt
# Kaggle's torchvision is built for its own torch and crashes on import with this one; transformers
# imports it whenever it is present, and the pipeline does not need it.
!pip uninstall -y -q torchvision
!python -I -c "from transformers import CLIPSegForImageSegmentation, SiglipModel; from diffusers import StableDiffusion3Pipeline; print('libraries ok')"
!git log -1 --oneline


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi -L


## 1. Verify before anything else

`test_processor.py` runs on CPU. `verify_checkpoint.py` checks the real weights and reports memory: look at `mem_guided_gb` before going on. Its output goes to `/kaggle/working/verify`.


In [ ]:
!python -I code/test_processor.py | tail -5
!mkdir -p /kaggle/working/verify
!python -I code/verify_checkpoint.py --model stabilityai/stable-diffusion-3-medium-diffusers --vae-device {VAE_DEVICE} --out /kaggle/working/verify | tail -30


## 1b. Check phrase tokens on the real T5 tokenizer

CPU tests use fake offsets, so this confirms each phrase maps to its own tokens. Each line should show the phrase it names.


In [ ]:
import sys
sys.path.insert(0, 'code')
from transformers import AutoTokenizer
from grounding import map_tokens, phrase_spans
from predicates import translate
tok = AutoTokenizer.from_pretrained('stabilityai/stable-diffusion-3-medium-diffusers', subfolder='tokenizer_3')
for prompt in ['a red mug and a red teapot', 'a greenish-gray mug and a turquoise-green teapot', 'a kitten slightly larger than a pillow']:
    enc = tok(prompt, padding='max_length', max_length=256, truncation=True, return_offsets_mapping=True)
    tokens, bad = map_tokens([tuple(o) for o in enc['offset_mapping']], phrase_spans(prompt, translate(prompt)))
    print(prompt, 'unmapped:', bad)
    for phrase, idx in tokens.items():
        print('  ', phrase, '->', tok.decode([enc['input_ids'][j - 77] for j in idx]))


In [ ]:
# Stops here if the check above failed: the rest of the notebook means nothing without it.
import json
check = json.load(open('/kaggle/working/verify/checkpoint_check.json'))
print({k: check.get(k) for k in ('mem_unguided_gb', 'mem_guided_gb', 'mem_guided_vae_gpu_gb', 'mem_sar_gb', 'identity_out', 'identity_grad', 'identity_grad_noise_floor', 'readout_grad', 'fd_grad', 'image_identity', 'token_index')})


## 1c. One two-object image through every stage

A fast check of memory and of the CLIPSeg, SigLIP, and SAR path on real weights, before any long run. `peak_by_gpu` in the printed output shows both GPUs.


In [ ]:
!python -I code/run_experiment.py --suite plain_color --group dev --n 1 --arms stage1,stage1_sar --out {OUT}/smoke --vae-device {VAE_DEVICE}
import glob
for p in glob.glob(f'{OUT}/smoke/*/*/meta.json'):
    m = json.load(open(p))
    print(m['arm'], '| failed:', m.get('failed'), '| time_s:', m.get('time_s'), '| peak_by_gpu:', m.get('peak_by_gpu'),
          '| bias_steps:', m.get('bias_steps'), '| no_op:', m.get('no_op_reason'), '| fallback:', m.get('used_fallback'))
    if m.get('traceback'):
        print(m['traceback'][-1500:])


## 1d. Measure presence and attention thresholds

Generates 20 unguided development images, then scores objects that are in each image against objects that are not, and checks where attention lands. Look at `{OUT}/probe/images`: the labels assume each image shows the objects its prompt names.


In [ ]:
!python -I code/threshold_probe.py --out {OUT}/probe --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn'


## 2. M1 tuning

Three step sizes, two points to start guidance (sigma 0.75 and 0.5), and two directions, on 5 separate prompts. Compare the images as well as the numbers: lightness can reach its target while the scene turns into something else.


In [ ]:
resume = f'--resume-from {RESUME}/m1_tune' if RESUME else ''
!python -I code/run_experiment.py --suite m1_tune --out {OUT}/m1_tune --vae-device {VAE_DEVICE} {resume}


Set the chosen `cfgEtaZero`, `cfgSigmaHi` and `cfgMaxPert` in `configs/provisional.yaml` before the next cell.


In [ ]:
import json, glob, collections
rows = collections.defaultdict(list)
for p in glob.glob(f'{OUT}/m1_tune/*@*/*/meta.json'):
    m = json.load(open(p))
    if 'dy' in m:
        rows[m['arm']].append((m['ds'], m['dy'], m['time_s']))
for arm, r in sorted(rows.items()):
    print(arm, 'n', len(r), 'mean dy/ds', round(sum(b / a for a, b, _ in r) / len(r), 3), 'time_s', round(sum(t for *_, t in r) / len(r), 1))


## 3. M1 evaluation and ladder


In [ ]:
resume = f'--resume-from {RESUME}/m1' if RESUME else ''
!python -I code/run_experiment.py --suite m1_eval --out {OUT}/m1 --vae-device {VAE_DEVICE} {resume}


## 4. Blind rating page

Download `rate.html`, rate on your own machine, and save the exported text as `ratings.json`. Don't open `rate.key.json` until you've finished rating.


In [ ]:
import sys, glob
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_rating_page
images = [Path(p) for p in glob.glob(f'{OUT}/m1/*/*/image.png') if '/ladder' not in p]
build_rating_page(images, Path(f'{OUT}/m1/rate.html'), seed=0)
print(len(images), 'images')


## 5. Colour, size, and SAR on development prompts

Debug on plain colours, then compare on mixtures. All arms use the same prompts and seeds. The last three arms refine the same Stage 1 image from the same re-noised latent.


In [ ]:
resume = f'--resume-from {RESUME}/plain' if RESUME else ''
!python -I code/run_experiment.py --suite plain_color --group dev --n 5 --out {OUT}/plain --vae-device {VAE_DEVICE} {resume}


In [ ]:
resume = f'--resume-from {RESUME}/mixture' if RESUME else ''
!python -I code/run_experiment.py --suite mixture --group dev --n 20 --out {OUT}/mixture --vae-device {VAE_DEVICE} {resume}


In [ ]:
resume = f'--resume-from {RESUME}/size' if RESUME else ''
!python -I code/run_experiment.py --suite size --group dev --n 10 --arms unguided,stage1 --out {OUT}/size --vae-device {VAE_DEVICE} {resume}


## 6. Blind SAR comparison

Download `compare.html` and rate it on your own machine. Don't open `compare.key.json` until you've finished. Check `fallback_rate` and `no_bias_share` too: a SAR run that always falls back or never applies its bias looks harmless while doing nothing.


In [ ]:
import importlib, json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_comparison_page
from run_experiment import summarize_arms
build_comparison_page(f'{OUT}/mixture', Path(f'{OUT}/mixture/compare.html'), seed=0)
print(json.dumps(summarize_arms(f'{OUT}/mixture'), indent=1))


## 7. Your own prompt

Runs one prompt through the whole pipeline and shows SD3 alone, Stage 1, and Stage 1 + SAR side by side. Results use provisional calibration unless `--calibration` is given, so they are for looking at, not for the paper.


In [ ]:
PROMPT = 'a slightly red mug and a blue teapot'
SEED = 0
!python -I code/run_experiment.py --prompt "{PROMPT}" --seeds {SEED} --out {OUT}/mine --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn'


In [ ]:
import hashlib, json
import matplotlib.pyplot as plt
from PIL import Image
image_id = f"{hashlib.sha256(PROMPT.encode()).hexdigest()[:12]}_s{SEED}"
arms = [('unguided', 'SD3 alone'), ('stage1', 'Stage 1'), ('stage1_sar', 'Stage 1 + SAR')]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (arm, title) in zip(axes, arms):
    d = f'{OUT}/mine/{arm}/{image_id}'
    m = json.load(open(f'{d}/meta.json'))
    if 'failed' in m:
        ax.set_title(f"{title}: failed ({m['failed']})")
    else:
        ax.imshow(Image.open(f'{d}/image.png'))
        note = ' (fell back to Stage 1)' if m.get('used_fallback') else ''
        ax.set_title(title + note)
    ax.axis('off')
plt.suptitle(PROMPT)
plt.show()


## 8. FMAS pilot

Ten soft-attribute development prompts through SD3, Stage 1, Stage 1 + plain pass, and Stage 1 + SAR. Download `fmas.html` and rate it on your own machine. Don't open `fmas.key.json` until you've finished. Author ratings with provisional calibration: a pilot, never paper evidence.


In [ ]:
!python -I code/run_experiment.py --suite soft --group dev --n 10 --arms unguided,stage1,stage1_plain,stage1_sar --out {OUT}/fmas --vae-device {VAE_DEVICE} 2>&1 | grep -v -E 'Loading|it/s|warn' | tail -30
import sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import build_fmas_page
build_fmas_page(f'{OUT}/fmas', Path(f'{OUT}/fmas/fmas.html'), seed=0)
print('rating page:', f'{OUT}/fmas/fmas.html')


In [ ]:
# Press "Show ratings" on the page, then paste the text between the quotes.
RATINGS = r'''PASTE HERE'''
import json, sys
sys.path.insert(0, 'code')
from pathlib import Path
from annotate import score_fmas
Path(f'{OUT}/fmas/ratings.json').write_text(RATINGS)
print(json.dumps(score_fmas(f'{OUT}/fmas/ratings.json', f'{OUT}/fmas/fmas.key.json'), indent=1))
